# Kmitanie mechanickej sústavy

Kmitanie je periodický pohyb telesa okolo rovnovážnej polohy.
V strojárstve sa stretávame s kmitaním pri vibráciách motorov,
podvozkov vozidiel, mostov či strojných súčastí. Pochopenie
tlmeného kmitania je kľúčové pri návrhu tlmičov a pružín.

V tomto notebooku riešime pohybovú rovnicu tlmeného harmonického
oscilátora numericky pomocou knižnice SciPy.

Pri otvorení v JupyterLab sú k dispozícii interaktívne slidery –
hodnoty môžeš meniť aj priamo v bloku PARAMETRE v kóde.

## Teoretický základ

Pohybová rovnica tlmeného harmonického oscilátora:

$$m \cdot \ddot{x} + b \cdot \dot{x} + k \cdot x = 0$$

kde:
- $m$ – hmotnosť telesa [kg]
- $b$ – koeficient tlmenia [N·s/m]
- $k$ – tuhosť pružiny [N/m]
- $x$ – výchylka od rovnovážnej polohy [m]

**Vlastná uhlová frekvencia:**
$$\omega_0 = \sqrt{\frac{k}{m}}$$

**Kritické tlmenie:**
$$b_{krit} = 2 \cdot \sqrt{k \cdot m}$$

Podľa pomeru $b$ a $b_{krit}$ rozlišujeme tri režimy:

| Režim | Podmienka | Popis |
|-------|-----------|-------|
| Podtlmený | $b < b_{krit}$ | Sústava kmitá s klesajúcou amplitúdou |
| Kriticky tlmený | $b = b_{krit}$ | Najrýchlejší návrat bez kmitania |
| Pretlmený | $b > b_{krit}$ | Pomalý návrat bez kmitania |

## Simulácia tlmeného oscilátora

Nasledujúci kód numericky rieši pohybovú rovnicu pomocou SciPy a vykreslí výchylku, rýchlosť, fázový diagram a porovnanie troch režimov tlmenia.

![Kmitanie mechanickej sústavy](_images/tlmeny1.png)
*Obr. Tlmený harmonický oscilátor – výchylka, rýchlosť, fázový diagram a porovnanie režimov*

In [2]:
# ============================================================
# AKO POUŽÍVAŤ TENTO NOTEBOOK V JUPYTERLAB:
# 1. Zmeň hodnoty v bloku PARAMETRE nižšie
# 2. Spusti bunku klávesou Shift+Enter
# 3. Alebo použi slidery ktoré sa zobrazia pod bunkou
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.integrate import solve_ivp
%matplotlib inline
plt.style.use('dark_background')

# ============================================================
# PARAMETRE – môžeš meniť tu ALEBO cez slidery nižšie
# ============================================================
m_def  = 1.0   # hmotnosť telesa [kg]
k_def  = 10.0  # tuhosť pružiny [N/m]
b_def  = 1.0   # koeficient tlmenia [N·s/m]  (0 = bez tlmenia)
x0_def = 1.0   # počiatočná výchylka [m]
# ============================================================

out = widgets.Output()

def simuluj_kmitanie(m, k, b, x0):
    # Vypočítame kritické tlmenie a vlastnú frekvenciu
    b_krit = 2 * np.sqrt(k * m)          # kritické tlmenie [N·s/m]
    omega0 = np.sqrt(k / m)               # vlastná uhlová frekvencia [rad/s]
    pomer  = b / b_krit                   # pomer aktuálneho a kritického tlmenia

    # Určenie režimu tlmenia
    if pomer < 0.99:
        rezim = 'PODTLMENÝ'
    elif pomer <= 1.01:
        rezim = 'KRITICKY TLMENÝ'
    else:
        rezim = 'PRETLMENÝ'

    # Pohybová rovnica: m·ẍ + b·ẋ + k·x = 0
    def pohybova_rovnica(t, y):
        x, v = y
        dxdt = v
        dvdt = (-b * v - k * x) / m      # z pohybovej rovnice
        return [dxdt, dvdt]

    # Časový rozsah simulácie (min. 5 s, inak 15 periód)
    T_kyvov = 2 * np.pi / omega0
    t_max   = max(15 * T_kyvov, 5)
    t_eval  = np.linspace(0, t_max, 2000)

    # Numerické riešenie pomocou Runge-Kutta RK45
    sol = solve_ivp(
        pohybova_rovnica,
        (0, t_max),
        [x0, 0.0],                        # počiatočné podmienky: x=x0, v=0
        t_eval=t_eval,
        method='RK45',
        rtol=1e-8
    )

    t = sol.t
    x = sol.y[0]
    v = sol.y[1]

    # Obálka exponenciálneho útlmu (len pre podtlmený režim)
    if pomer < 1:
        alpha  = b / (2 * m)              # koeficient útlmu
        obalka = x0 * np.exp(-alpha * t)

    with out:
        clear_output(wait=True)

        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(13, 10))
        fig.patch.set_facecolor('#0d0d1a')

        # ── Graf výchylky ─────────────────────────────────────
        ax1.set_facecolor('#0d0d1a')
        ax1.plot(t, x, color='#00BFFF', linewidth=2.2, label='Výchylka x(t)')
        ax1.axhline(y=0, color='gray', linewidth=1, linestyle='--', alpha=0.3)
        if pomer < 1:
            ax1.plot(t,  obalka, color='#FFD700', linewidth=1.5,
                     linestyle='--', alpha=0.7, label='Obálka útlmu')
            ax1.plot(t, -obalka, color='#FFD700', linewidth=1.5,
                     linestyle='--', alpha=0.7)
        ax1.fill_between(t, x, 0, where=(x >= 0), alpha=0.08, color='#00BFFF')
        ax1.fill_between(t, x, 0, where=(x < 0),  alpha=0.08, color='#FF6B6B')
        ax1.set_xlabel('Čas t (s)', color='#cccccc')
        ax1.set_ylabel('Výchylka x (m)', color='#cccccc')
        ax1.set_title('Závislosť výchylky od času', color='white', fontsize=11)
        ax1.legend(fontsize=9, facecolor='#1a1a2e', edgecolor='#555555')
        ax1.grid(True, alpha=0.12)
        ax1.tick_params(colors='#aaaaaa')

        # ── Fázový diagram ────────────────────────────────────
        ax2.set_facecolor('#0d0d1a')
        body = ax2.scatter(x, v, c=t, cmap='plasma', s=1.5, alpha=0.8)
        ax2.axhline(y=0, color='gray', lw=0.8, alpha=0.3)
        ax2.axvline(x=0, color='gray', lw=0.8, alpha=0.3)
        ax2.set_xlabel('Výchylka x (m)', color='#cccccc')
        ax2.set_ylabel('Rýchlosť v (m/s)', color='#cccccc')
        ax2.set_title('Fázový diagram', color='white', fontsize=11)
        ax2.grid(True, alpha=0.12)
        ax2.tick_params(colors='#aaaaaa')
        cb = plt.colorbar(body, ax=ax2)
        cb.set_label('Čas (s)', color='#cccccc')
        cb.ax.yaxis.set_tick_params(color='#aaaaaa')

        # ── Graf rýchlosti ────────────────────────────────────
        ax3.set_facecolor('#0d0d1a')
        ax3.plot(t, v, color='#FFA500', linewidth=2.0, label='Rýchlosť v(t)')
        ax3.axhline(y=0, color='gray', linewidth=1, linestyle='--', alpha=0.3)
        ax3.fill_between(t, v, 0, where=(v >= 0), alpha=0.08, color='#FFA500')
        ax3.fill_between(t, v, 0, where=(v < 0),  alpha=0.08, color='#FF6B6B')
        ax3.set_xlabel('Čas t (s)', color='#cccccc')
        ax3.set_ylabel('Rýchlosť v (m/s)', color='#cccccc')
        ax3.set_title('Závislosť rýchlosti od času', color='white', fontsize=11)
        ax3.legend(fontsize=9, facecolor='#1a1a2e', edgecolor='#555555')
        ax3.grid(True, alpha=0.12)
        ax3.tick_params(colors='#aaaaaa')

        # ── Porovnanie troch režimov tlmenia ──────────────────
        ax4.set_facecolor('#0d0d1a')
        t_por = np.linspace(0, t_max, 2000)
        for b_por, farba, nazov, ls in [
            (b_krit * 0.3, '#00FF88', 'Podtlmený', '--'),
            (b_krit * 1.0, '#FFD700', 'Kritický',   '-'),
            (b_krit * 3.0, '#FF6B6B', 'Pretlmený', ':'),
        ]:
            sol_por = solve_ivp(
                lambda t, y, b_p=b_por: [y[1], (-b_p * y[1] - k * y[0]) / m],
                (0, t_max), [x0, 0.0],
                t_eval=t_por, method='RK45'
            )
            ax4.plot(sol_por.t, sol_por.y[0],
                     color=farba, linewidth=1.8,
                     linestyle=ls, label=nazov, alpha=0.85)
        ax4.axhline(y=0, color='gray', lw=0.8, alpha=0.3)
        ax4.set_xlabel('Čas t (s)', color='#cccccc')
        ax4.set_ylabel('Výchylka x (m)', color='#cccccc')
        ax4.set_title('Porovnanie režimov tlmenia', color='white', fontsize=11)
        ax4.legend(fontsize=8.5, facecolor='#1a1a2e', edgecolor='#555555')
        ax4.grid(True, alpha=0.12)
        ax4.tick_params(colors='#aaaaaa')

        fig.suptitle(
            f'Tlmený harmonický oscilátor  │  m = {m} kg  │  '
            f'k = {k} N/m  │  b = {b:.1f} N·s/m  │  Režim: {rezim}',
            fontsize=12, color='white'
        )
        plt.tight_layout()
        plt.show()

        # Výpis vypočítaných hodnôt
        print(f'Vlastná frekvencia:    {omega0:.3f} rad/s  ({omega0/(2*np.pi):.3f} Hz)')
        print(f'Kritické tlmenie:      {b_krit:.2f} N·s/m')
        print(f'Pomer b/b_krit:        {pomer:.3f}')
        print(f'Režim tlmenia:         {rezim}')

_ = widgets.interact(
    simuluj_kmitanie,
    m=widgets.FloatSlider(
        value=m_def, min=0.5, max=5.0, step=0.1,
        description='m (kg):',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='500px')),
    k=widgets.FloatSlider(
        value=k_def, min=1.0, max=50.0, step=1.0,
        description='k (N/m):',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='500px')),
    b=widgets.FloatSlider(
        value=b_def, min=0.0, max=20.0, step=0.5,
        description='b (N·s/m):',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='500px')),
    x0=widgets.FloatSlider(
        value=x0_def, min=0.1, max=2.0, step=0.1,
        description='x₀ (m):',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='500px'))
)
display(out)

interactive(children=(FloatSlider(value=1.0, description='m (kg):', layout=Layout(width='500px'), max=5.0, min…

Output()